<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Homographies

Le modèle de caméra sténopé établit une correspondance plusieurs-à-un entre les points de la scène et leur projection sur le plan image. Connaissant les paramètres intrinsèques et extrinsèques qui définissent la matrice de la caméra, nous pouvons relier les points de l'image à leurs rayons correspondants dans la scène. Cependant, nous ignorons la position du point correspondant sur ce rayon.

Une façon de lever cette ambiguïté consiste à utiliser la connaissance préalable de la géométrie de l'environnement pour structurer davantage le modèle de projection. Dans le cas de Duckietown, nous exploiterons la planéité de l'environnement. Dans d'autres contextes, comme celui des véhicules autonomes, même si le monde n'est pas globalement plan, nous pouvons le considérer comme localement plan autour du robot (par exemple, imaginons le plan tangent à la surface courbe sur laquelle le robot se déplace). Nous pouvons alors tirer parti de cette planéité pour contraindre le modèle de projection.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/homography/image-ground-projection.png", width="400px" />
  <p>Une homographie fournit une transformation inversible entre les coordonnées homogènes de points situés dans deux plans.</p>
  </div>
</figure>

Considérons le scénario ci-dessus où le monde est plan et constitué d'un ensemble de points $\mathbf{X}_i, \; i \in \{1,2,\ldots,n\}$, exprimés en coordonnées homogènes. Ces points se projettent sur une image de la scène et deviennent les points $\mathbf{x}_i$. Chaque paire de points peut être reliée par la matrice de projection de la caméra.

$$
\mathbf{x}_i = 
\begin{bmatrix}
f_x & s & p_x\\
0 & f_y & p_y\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
\mathbf{R} \; \vert \; \mathbf{t}
\end{bmatrix} \mathbf{X}_i
$$

où $\mathbf{R}$ et $\mathbf{t}$ sont respectivement la matrice de rotation et le vecteur de translation qui définissent la transformation du repère monde au repère caméra. Puisque nous sommes libres de définir le repère monde comme bon nous semble, définissons-le de sorte que le plan monde soit situé dans le plan $x-y$, l'axe $z$ étant orthogonal et orienté vers le haut. Nous pouvons alors écrire l'opération de projection comme suit :

$$
\begin{align}
\mathbf{x}_i &= 
\begin{bmatrix}
f_x & s & p_x\\
0 & f_y & p_y\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
R_{11} & R_{12} & R_{13} & t_x\\
R_{21} & R_{22} & R_{23} & t_y\\
R_{31} & R_{32} & R_{33} & t_z\\
\end{bmatrix} 
\begin{bmatrix}
X_i\\
Y_i\\
Z_i\\
1
\end{bmatrix}\\
&= 
\begin{bmatrix}
f_x & s & p_x\\
0 & f_y & p_y\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
R_{11} & R_{12} & R_{13} & t_x\\
R_{21} & R_{22} & R_{23} & t_y\\
R_{31} & R_{32} & R_{33} & t_z\\
\end{bmatrix} 
\begin{bmatrix}
X_i\\
Y_i\\
0\\
1
\end{bmatrix} \quad \textrm{since } Z_i = 0\\
&=
\begin{bmatrix}
f_x & s & p_x\\
0 & f_y & p_y\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
R_{11} & R_{12} & t_x\\
R_{21} & R_{22} & t_y\\
R_{31} & R_{32} & t_z\\
\end{bmatrix} 
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix}\\
&=
\begin{bmatrix}
H_{11} & H_{12} & H_{13}\\
H_{21} & H_{22} & H_{23}\\
H_{31} & H_{32} & H_{33}\\
\end{bmatrix} 
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix}
\end{align}
$$

La matrice $H$ définit une transformation projective et est appelée *homographie*. La matrice est de rang maximal et définit une application inversible entre les points du monde (dans ce cas uniquement les points du plan du sol) et leur projection correspondante dans l'image, c'est-à-dire

$$ 
\begin{align}
\mathbf{x}_i &= H 
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix}\\
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix} &= H^{-1} \mathbf{x}_i
\end{align}
$$

Bien que la matrice d'homographie possède 9 éléments, on peut multiplier $H$ par n'importe quelle constante non nulle sans modifier la projection. Pour le démontrer, considérons la transformation suivante impliquant des coordonnées d'image non homogènes : $[x_i \; y_i]^\top$

$$
\begin{bmatrix}
x_i\\
y_i\\
1
\end{bmatrix} = H
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix}
$$

Multiplying $H$ by a non-zero constant $\lambda$ yields

$$
\begin{bmatrix}
\lambda x_i\\
\lambda y_i\\
\lambda
\end{bmatrix} = (\lambda H)
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix}
$$

Comme il s'agit de coordonnées homogènes, le point projeté possède les mêmes coordonnées non homogènes $[x_i \; y_i]^\top$. ​​Ainsi, bien que la matrice d'homographie comporte 9 valeurs, elle ne possède que 8 degrés de liberté.

### Exemple

Les homographies définissent une projection d'un plan sur un autre. Nous avons décrit cela précédemment dans le contexte d'une projection $H$ des points situés sur un plan du monde vers le plan image pour une pose de caméra donnée. L'homographie établit une correspondance biunivoque entre les points du plan du monde et leur projection correspondante dans l'image. Si l'on déplaçait la caméra, on obtiendrait une homographie $\bar{H}$ différente reliant l'image au plan du sol.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/homography/image-ground-image-projection.png", width="600px" />
  <p>A visualization of a simple pinhole camera.</p>
  </div>
</figure>

Nous pouvons composer ces homographies pour obtenir une homographie qui relie les coordonnées homogènes des points d'une image à leurs coordonnées correspondantes dans la seconde image.

$$
\begin{align}
\mathbf{x}_i &= H \mathbf{X}_i\\
&= H\left(\bar{H}^{-1}\mathbf{x}^\prime_i\right)\\
&= \left(H\bar{H}^{-1}\right)\mathbf{x}^\prime_i\\
&= \check{H}\mathbf{x}^\prime_i
\end{align}
$$

où $\check{H} = H\bar{H}^{-1}$ est l'homographie entre les deux images qui est induite par le plan du sol.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/homography/bt.000.png", width=400px>
  <img src="../../assets/images/homography/bt.002.png", width=400px>
  <p>Deux images d'un couloir du campus d'Oxford prises de points de vue différents. La planéité du sol induit une homographie $\check{H}$ entre les deux images.</p>
  </div>
</figure>

Dans cet exemple, vous explorerez l'utilisation d'une homographie pour relier des images d'une même scène prises sous différents angles. Bien que la scène ne soit pas plane, certains plans permettent d'induire une homographie entre les deux vues. Dans l'exemple ci-dessous, le plan du sol a été utilisé pour estimer l'homographie. Dans cet exercice, vous examinerez la validité de cette homographie et vérifierez si elle fournit une transformation valide entre des points qui ne se trouvent pas sur le sol.

Pensez-vous que la transformation sera valide pour des points situés sur les murs ? Pourquoi ? (Attention en particulier au mur de droite, le plus proche de la caméra.)

In [ ]:
### Exécutez cette cellule pour importer les modules pertinents
%matplotlib widget

import matplotlib
import numpy as np
import cv2
from matplotlib import pyplot as plt

In [ ]:
# TODO: L'exécution de cette cellule affichera une figure présentant les deux images de la scène ci-dessus. Lorsque vous cliquez sur un point de l'image de gauche,
#. le point projeté dans l'image de droite sera rendu selon l'homographie induite par le plan du sol.
#. Comparez la précision des correspondances entre les points situés sur le plan du sol et les points situés ailleurs dans l'environnement.
imgl = cv2.imread('../../assets/images/homography/bt.000.png', 0)
imgr = cv2.imread('../../assets/images/homography/bt.002.png', 0)

H = np.array([[0.907503833504229, -0.116496578881938, 30.8471918181923],[0.00308072860216055, 0.828815989469247, 16.0448537015201],[-1.74015013507422e-05, -0.000441721032603193, 1]])

fig = plt.figure(figsize = (12,6))
ax1 = fig.add_subplot(1,2,1)
ax1.imshow(imgl,cmap = 'gray')
ax1.set_title('Source Image')
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(imgr,cmap = 'gray')
ax2.set_title('Target Image')

def onclick(event):
    # Appliquer l'homographie
    x = np.array([[event.xdata, event.ydata, 1]]).transpose()
    xprime = H.dot(x)
    xprime = xprime/xprime[2]
    
    # Visualisez les points sélectionnés et projetés
    ax1.plot(x[0], x[1], 'rx')
    ax2.plot(xprime[0], xprime[1], 'rx')

cid = fig.canvas.mpl_connect('button_press_event', onclick)

Vous pouvez maintenant passer au [notebook sur le filtrage d'images](../03-Image-Filtering/image_filtering.ipynb).